# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\notebooks
프로젝트 루트: c:\Users\ojh23\Documents\projects\llm-data-analysis-course
전처리 데이터 폴더: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed
보고서 폴더: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [3]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [5]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [6]:
customers['age'].describe()


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [7]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


In [8]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,F,84,56.0
1,M,66,44.0


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [9]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


In [10]:
products['price'].describe()


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [11]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [12]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


In [13]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


In [14]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00
주문 월 개수: 13


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [15]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [16]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (764, 6)
병합 후 sales_items: (764, 9)
상품명 누락: 0
카테고리 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,100,3,102000,306000,도서 상품 100,도서,102000
1,2,1,87,5,25000,125000,도서 상품 087,도서,25000
2,3,1,7,3,142000,426000,도서 상품 007,도서,142000
3,4,1,9,3,193000,579000,스포츠 상품 009,스포츠,193000
4,5,2,72,4,189000,756000,뷰티 상품 072,뷰티,189000


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [17]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14
0,도서,238,24645000,9.64
6,패션,220,23801000,9.31


In [18]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,468,50174000,19.63,111579.0
1,뷰티,376,47551000,18.60,117688.0
2,전자기기,401,41003000,16.04,101588.0
3,생활용품,390,34839000,13.63,96438.0
4,식품,240,33597000,13.14,137143.0
5,도서,238,24645000,9.64,106857.0
6,패션,220,23801000,9.31,115909.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [19]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (764, 12)
order_date 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,100,3,102000,306000,123,2026-05-07,card,completed,2026-05,Thursday
1,2,1,87,5,25000,125000,123,2026-05-07,card,completed,2026-05,Thursday
2,3,1,7,3,142000,426000,123,2026-05-07,card,completed,2026-05,Thursday
3,4,1,9,3,193000,579000,123,2026-05-07,card,completed,2026-05,Thursday
4,5,2,72,4,189000,756000,77,2025-07-23,naver_pay,cancelled,2025-07,Wednesday


In [20]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,11827000,15,788467.0
1,2025-08,21134000,24,880583.0
2,2025-09,19275000,23,838043.0
3,2025-10,38555000,39,988590.0
4,2025-11,19961000,23,867870.0
5,2025-12,21909000,26,842654.0
6,2026-01,22002000,29,758690.0
7,2026-02,22953000,29,791483.0
8,2026-03,21646000,21,1030762.0
9,2026-04,23112000,31,745548.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [21]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


In [29]:
# completed 주문 Gate를 적용한 금액성 EDA
# 공식 Notebook 12~14번의 전체 주문 집계와 구분해 사용합니다.

# 1) order_items + orders 병합 후 검증
orders_for_sales = orders[
    ["order_id", "customer_id", "order_date", "order_status"]
].copy()

print("orders.order_id 중복:", orders_for_sales["order_id"].duplicated().sum())

completed_items = order_items.merge(
    orders_for_sales,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전 order_items 행 수:", len(order_items))
print("병합 후 행 수:", len(completed_items))
display(completed_items["_merge"].value_counts())

assert len(completed_items) == len(order_items)
assert completed_items["_merge"].eq("both").all()

completed_items = completed_items.drop(columns="_merge")

# 2) completed 주문만 필터
completed_items = completed_items.loc[
    completed_items["order_status"].eq("completed")
].copy()

print("전체 주문 상세 행:", len(order_items))
print("completed 상세 행:", len(completed_items))
print("completed 주문 수:", completed_items["order_id"].nunique())
print("completed 주문 기준 금액:", completed_items["line_total"].sum())

# 3) 상품 병합 후 카테고리별 집계
completed_items = completed_items.merge(
    products[["product_id", "product_name", "category", "price"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

display(completed_items["_merge"].value_counts())
assert completed_items["_merge"].eq("both").all()

completed_items = completed_items.drop(columns="_merge")

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

category_sales["sales_ratio"] = (
    category_sales["total_sales"] / category_sales["total_sales"].sum() * 100
).round(2)

# 4) 월별 집계
completed_items["order_month"] = (
    pd.to_datetime(completed_items["order_date"])
    .dt.to_period("M")
    .astype(str)
)

monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
    )
    .sort_values("order_month")
)

monthly_sales["avg_order_value"] = (
    monthly_sales["total_sales"] / monthly_sales["order_count"]
).round(0)

# 5) 고객별 집계: 이름은 포함하지 않음
customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

customer_sales = customer_sales.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)

customer_sales["avg_order_value"] = (
    customer_sales["total_sales"] / customer_sales["order_count"]
).round(0)

# 6) 같은 completed 범위의 총합 검증
source_total = completed_items["line_total"].sum()

total_check = pd.DataFrame({
    "집계 기준": [
        "completed 원본 line_total",
        "카테고리별 total_sales 합계",
        "월별 total_sales 합계",
        "고객별 total_sales 합계",
    ],
    "total_sales": [
        source_total,
        category_sales["total_sales"].sum(),
        monthly_sales["total_sales"].sum(),
        customer_sales["total_sales"].sum(),
    ],
})

total_check["원본과의 차이"] = total_check["total_sales"] - source_total
total_check["일치 여부"] = total_check["원본과의 차이"].eq(0)

display(category_sales)
display(monthly_sales)
display(customer_sales.head(10))
display(total_check)

assert total_check["일치 여부"].all()

orders.order_id 중복: 0
병합 전 order_items 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

전체 주문 상세 행: 764
completed 상세 행: 474
completed 주문 수: 184
completed 주문 기준 금액: 148990000


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,order_month,total_sales,order_count,avg_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


,customer_id,order_count,total_sales,city,avg_order_value
0,117,5,4100000,성남,820000.0
1,102,4,3996000,고양,999000.0
2,83,4,3880000,수원,970000.0
3,30,5,3590000,서울,718000.0
4,40,4,3523000,서울,880750.0
5,20,2,3191000,인천,1595500.0
6,3,2,3178000,성남,1589000.0
7,111,3,3153000,광주,1051000.0
8,66,4,3093000,서울,773250.0
9,147,2,2990000,부산,1495000.0


,집계 기준,total_sales,원본과의 차이,일치 여부
0,completed 원본 line_total,148990000,0,True
1,카테고리별 total_sales 합계,148990000,0,True
2,월별 total_sales 합계,148990000,0,True
3,고객별 total_sales 합계,148990000,0,True


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [22]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [23]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/reports/ch06_category_sales.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/reports/ch06_customer_city.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/reports/ch06_customer_sales.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/reports/ch06_eda_questions.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/reports/ch06_monthly_sales.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/reports/ch06_product_category.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [24]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.

> 주의: 이 저장소의 `src/eda.py` 출력은 기존 전체 주문 기준 구현입니다. 본 제출 답안의 금액성 EDA는 14절 아래에서 직접 검증한 `order_status == "completed"` 기준 결과를 사용합니다.


In [25]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [26]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [35]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
# 과제 1. 도시별 고객 수와 도시별 completed 주문 기준 구매 금액 비교

city_customer_count = (
    customers.groupby("city", as_index=False)
    .agg(customer_count=("customer_id", "nunique"))
)

city_sales = (
    completed_items
    .merge(
        customers[["customer_id", "city"]],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .groupby("city", as_index=False)
    .agg(
        completed_total_sales=("line_total", "sum"),
        completed_order_count=("order_id", "nunique"),
    )
)

city_comparison = (
    city_customer_count
    .merge(city_sales, on="city", how="left", validate="one_to_one")
    .fillna({"completed_total_sales": 0, "completed_order_count": 0})
    .sort_values("completed_total_sales", ascending=False)
)

display(city_comparison)

,city,customer_count,completed_total_sales,completed_order_count
6,성남,21,24099000,31
0,고양,11,19116000,18
5,서울,15,19097000,29
8,울산,14,15380000,20
1,광주,17,14264000,17
4,부산,16,13604000,13
9,인천,14,12832000,18
3,대전,14,11792000,14
7,수원,13,10620000,14
2,대구,15,8186000,10


In [36]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
# 과제 2. 상품 수가 많은 카테고리가 completed 주문 기준 금액도 높은지 확인

category_product_count = (
    products.groupby("category", as_index=False)
    .agg(product_count=("product_id", "nunique"))
)

category_comparison = (
    category_product_count
    .merge(
        category_sales[["category", "total_sales", "total_quantity"]],
        on="category",
        how="left",
        validate="one_to_one",
    )
    .sort_values("total_sales", ascending=False)
)

display(category_comparison)

print(
    "상품 수와 completed 주문 기준 금액의 상관계수:",
    round(category_comparison["product_count"].corr(category_comparison["total_sales"]), 3),
)

,category,product_count,total_sales,total_quantity
3,스포츠,19,31743000,295
5,전자기기,17,26400000,259
2,생활용품,16,23915000,272
1,뷰티,16,23383000,223
4,식품,7,16573000,133
0,도서,14,16389000,149
6,패션,11,10587000,111


상품 수와 completed 주문 기준 금액의 상관계수: 0.813


In [37]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
# 과제 3. 결제수단별 completed 주문 기준 평균 주문 금액

completed_order_amount = (
    completed_items
    .groupby("order_id", as_index=False)
    .agg(order_total=("line_total", "sum"))
)

payment_order_amount = completed_order_amount.merge(
    orders[["order_id", "payment_method"]],
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert payment_order_amount["_merge"].eq("both").all()
payment_order_amount = payment_order_amount.drop(columns="_merge")

payment_summary = (
    payment_order_amount
    .groupby("payment_method", as_index=False)
    .agg(
        completed_order_count=("order_id", "nunique"),
        average_order_value=("order_total", "mean"),
        total_sales=("order_total", "sum"),
    )
    .sort_values("average_order_value", ascending=False)
)

payment_summary["average_order_value"] = (
    payment_summary["average_order_value"].round(0)
)

display(payment_summary)

,payment_method,completed_order_count,average_order_value,total_sales
3,naver_pay,51,852941.0,43500000
1,card,39,815538.0,31806000
2,kakao_pay,49,802898.0,39342000
0,bank_transfer,45,763156.0,34342000


In [38]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
# 과제 4. 월별·카테고리별 completed 주문 기준 금액 요약

monthly_category_sales = (
    completed_items
    .groupby(["order_month", "category"], as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        total_quantity=("quantity", "sum"),
        order_count=("order_id", "nunique"),
    )
    .sort_values(["order_month", "total_sales"], ascending=[True, False])
)

monthly_category_pivot = (
    monthly_category_sales
    .pivot(index="order_month", columns="category", values="total_sales")
    .fillna(0)
)

display(monthly_category_sales)
display(monthly_category_pivot)

,order_month,category,total_sales,total_quantity,order_count
4,2025-07,전자기기,2328000,20,5
1,2025-07,뷰티,1336000,8,2
2,2025-07,생활용품,1117000,13,4
0,2025-07,도서,656000,7,3
3,2025-07,스포츠,432000,4,3
...,...,...,...,...,...
81,2026-07,식품,923000,9,1
79,2026-07,생활용품,560000,5,1
78,2026-07,도서,342000,3,1
80,2026-07,스포츠,218000,2,1


category,도서,뷰티,생활용품,스포츠,식품,전자기기,패션
order_month,,,,,,,
2025-07,656000.0,1336000.0,1117000.0,432000.0,0.0,2328000.0,0.0
2025-08,528000.0,2561000.0,1513000.0,3613000.0,2635000.0,2760000.0,2011000.0
2025-09,844000.0,1573000.0,1250000.0,2815000.0,268000.0,2696000.0,744000.0
2025-10,2550000.0,4606000.0,6027000.0,5799000.0,1899000.0,4118000.0,767000.0
2025-11,870000.0,1317000.0,867000.0,1611000.0,1271000.0,2272000.0,604000.0
2025-12,1975000.0,2305000.0,1060000.0,1103000.0,2130000.0,2928000.0,0.0
2026-01,2352000.0,1497000.0,2138000.0,5966000.0,2083000.0,2192000.0,1195000.0
2026-02,1044000.0,1794000.0,918000.0,2656000.0,1597000.0,464000.0,1276000.0
2026-03,990000.0,1385000.0,2100000.0,3768000.0,2237000.0,1005000.0,1944000.0


### 과제 5. 고객 유형 구분 기준

- 반복 구매 고객: completed 주문 기준 `order_count >= 2`
- 고액 단발 구매 고객: `order_count == 1`이면서 단일 주문 금액이
  단발 구매 고객의 75백분위수 이상인 고객

이 기준은 고객의 실제 가치나 충성도를 확정하는 기준이 아니라,
추가 분석을 위한 탐색적 구분 기준이다. 구매 주기, 취소·환불,
고객 유입 시점 등은 현재 데이터만으로 충분히 판단할 수 없다.

### 과제 6. LLM 프롬프트

현재 데이터에는 할인·프로모션·광고 노출 정보가 없습니다.
“특정 월에 매출이 증가한 원인이 무엇인가?”처럼 현재 데이터만으로
답할 수 없는 질문을, 현재 보유한 컬럼으로 검증 가능한 EDA 질문으로
바꿔 주세요.

보유 컬럼:
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total
- products: product_id, category, price
- customers: customer_id, gender, age, city

조건:
- 원인을 단정하지 말 것
- completed 주문 기준으로 계산 가능한 질문을 제안할 것
- 필요한 컬럼과 계산 지표를 함께 제시할 것

# Chapter 06 제출 답안 — 데이터를 보며 질문을 만드는 EDA

## 제출 정보

- 이름: 오병희
- GitHub ID: OBHAR
- 작성일: 2026-10-10
- 최종 Notebook URL: `https://github.com/OBHAR/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb`

## 1. 내가 정한 EDA 질문

| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 어떤 카테고리의 completed 주문 기준 금액이 큰가? | `order_status == "completed"` 주문 상세 | `total_sales`, `total_quantity`, 금액 비중 | `order_items`와 `orders`를 병합해 상태를 붙인 뒤 `products`와 병합한다. 카테고리 합계를 원본 합계와 비교한다. |
| 월별 completed 주문 기준 금액은 주문 수와 평균 주문 금액 중 무엇과 함께 변하는가? | completed 주문 | `total_sales`, `order_count`, `avg_order_value` | `order_id.nunique()`로 주문 수를 계산하고 월별 합계를 원본 합계와 비교한다. |
| 구매 금액이 큰 고객은 반복 구매 고객인가, 고액 단발 구매 고객인가? | completed 주문 | `order_count`, `total_sales`, `avg_order_value` | 고객별 집계 후 주문 횟수와 평균 주문 금액을 함께 비교한다. |

### 왜 이 질문을 선택했는가?

고객·상품·주문 상세가 분리된 구조이므로, 병합 이후에도 같은 주문 범위가 유지되는지 확인하면서 카테고리·시간·고객 관점의 질문을 연결해 보고자 했다. 금액은 `quantity × unit_price`인 `line_total`의 합계로 정의했으며, `completed` 주문만 포함했다.

## 2. 핵심 집계 결과

- completed 원본 합계: 148,990,000
- 카테고리별 합계: 148,990,000
- 월별 합계: 148,990,000
- 고객별 합계: 148,990,000
- 총합 일치 여부: 일치 (`원본과의 차이 = 0`)

![completed 주문 Gate 및 카테고리 결과](images/step02_completed_gate.png)

![completed 기준 총합 검증](images/step02_total_check.png)

### 결과 관찰

- `order_items` 764행을 `orders`와 병합한 뒤에도 764행이 유지됐고, 미매칭은 0건이었다.
- completed 주문은 184건, 해당 주문 상세 행은 474건이었다.
- 스포츠 카테고리의 completed 주문 기준 금액은 31,743,000원으로 가장 컸고, 전체의 21.31%였다.
- 카테고리·월별·고객별 `total_sales` 합계가 원본 completed `line_total` 합계와 모두 일치했다.

### 나의 해석과 판단

스포츠는 이번 분석 범위에서 금액 기여가 가장 큰 카테고리다. 그러나 금액이 가장 크다는 사실만으로 가장 인기 있거나 가장 수익성이 높다고 판단할 수는 없다. 판매 수량, 평균 단가, 상품 수를 함께 확인해야 하며, 본 결과는 completed 주문에 속한 상품 상세 금액의 합계로 해석한다.

### 업무·분석적 의미

카테고리·월별·고객별 집계가 같은 원본 합계로 검증되면, 이후 시각화나 추가 분석에서 어느 집계표를 사용하더라도 동일한 분석 범위를 유지할 수 있다. 또한 카테고리별 금액과 수량을 함께 보면 상품 운영 또는 재고 점검의 우선순위를 정하는 기초 자료가 될 수 있다.

### 한계와 추가 확인 사항

`line_total`은 할인, 배송비, 세금, 부분 환불, 정산 시점을 반영한 회계상 순매출이 아니다. 주문 기간은 2025-07-09부터 2026-07-08까지여서 2025년 7월과 2026년 7월은 완전한 한 달이 아닐 수 있다.

## 3. 관찰 → 가설 → 추가 검증

### 사례 1: 카테고리별 금액

- 관찰: 스포츠의 completed 주문 기준 금액은 31,743,000원으로 가장 크고, 판매 수량도 295개로 가장 많았다.
- 가설: 스포츠의 높은 금액은 판매 수량이 많거나 평균 판매 단가가 높기 때문일 수 있다.
- 추가 검증: 카테고리별 평균 판매 단가와 상품 수를 함께 비교하고, 특정 상품 몇 개가 금액을 크게 만들었는지 상품별 집계를 확인한다.

### 사례 2: 월별 금액

- 관찰: 2025년 10월의 completed 주문 기준 금액은 25,766,000원, 주문 수는 26건으로 월별 결과 중 가장 컸다.
- 가설: 해당 월의 금액 증가는 주문 수 증가 또는 평균 주문 금액 증가와 함께 나타났을 수 있다.
- 추가 검증: 2025년 10월을 다른 월과 비교하여 주문 수·평균 주문 금액·카테고리별 금액 구성을 함께 확인한다. 프로모션이나 계절성이 원인이라고는 현재 데이터만으로 단정하지 않는다.

### 사례 3: 고객별 구매 금액

- 관찰: 고객 ID 117의 completed 주문 기준 구매 금액은 4,100,000원이며 주문 수는 5건이었다.
- 가설: 높은 구매 금액이 반복 구매와 관련될 수 있다.
- 추가 검증: 고객별 `order_count`와 `avg_order_value`를 함께 비교하고, 반복 구매 고객과 고액 단발 구매 고객을 별도 기준으로 구분한다.

## 4. LLM 질문 확장

- Safe Context: 파일별 컬럼명, `completed` 분석 범위, 병합 키와 집계 결과만 전달했다. 고객 이름이나 원본 주문 행은 제공하지 않았다.
- Prompt: 현재 데이터로 검증 가능한 EDA 질문 10개, 필요한 데이터셋과 pandas 기능, 계산 가능한 지표를 요청했다. 데이터에 없는 내용은 추측하지 말 것을 조건으로 넣었다.
- LLM 제안 요약: 성별·연령대·도시별 고객 수, 주문 상태·결제수단별 주문 수, 월별 주문 수, 주문·상품·고객별 금액 분석 질문을 제안했다.
- 실제 사용 여부: 수정 후 사용
- 판단 이유: 고객·상품·주문 분포 질문은 실제 컬럼으로 계산 가능하다. 다만 금액성 질문은 `order_items`만으로 합산하지 않고 `orders`와 `order_id` 기준으로 병합한 뒤 `order_status == "completed"` 범위를 적용해야 한다.
- 사람이 수정한 내용: 금액성 집계에 completed 주문 Gate, `many_to_one` 병합 검증, 미매칭 확인, 원본·카테고리·월별·고객별 총합 일치 검증을 추가했다.

![LLM EDA 질문 확장](images/step04_llm_questions.png)

![답할 수 없는 질문의 재구성](images/step04_llm_reframe.png)

## 5. 다음 분석 우선순위

1. 월별 금액 변화가 주문 수·평균 주문 금액·카테고리 구성 중 무엇과 함께 변했는지 확인한다.
2. 스포츠 카테고리의 높은 금액이 판매 수량, 평균 단가, 특정 상품 중 무엇과 더 관련 있는지 확인한다.
3. 고객별 주문 횟수와 평균 주문 금액을 기준으로 반복 구매 고객과 고액 단발 구매 고객을 탐색적으로 구분한다.

### 가장 먼저 확인하고 싶은 이유

월별 금액은 주문 수와 평균 주문 금액이라는 서로 다른 요인의 결과일 수 있다. 이 둘을 먼저 분리해야 특정 월의 금액 차이를 과도하게 해석하지 않고 다음 분석 질문을 구체화할 수 있다.

### 현재 데이터만으로 단정할 수 없는 것

프로모션·광고·재고·배송·할인·원가·고객 만족도 정보가 없으므로, 특정 월이나 카테고리의 금액 변화 원인, 수익성, 고객 충성도를 확정할 수 없다. 또한 첫 달과 마지막 달은 부분 월일 수 있어 단순 월간 비교에 주의가 필요하다.

## 최종 체크

- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [x] 최종 Notebook URL을 LMS에 제출합니다.


## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.
